In [1]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [2]:
import os

# Ensure we are always in the correct project directory
DRIVE_DIR = "/content/drive/MyDrive/FundGitHubProject"
if os.path.exists(DRIVE_DIR):
    os.chdir(DRIVE_DIR)
    print("✅ Successfully moved to project directory:", os.getcwd())
else:
    print("❌ Directory not found. Please ensure your Drive is mounted and the folder exists.")

✅ Successfully moved to project directory: /content/drive/MyDrive/FundGitHubProject


In [ ]:
# Pull the latest updates for your branch before starting work
!python gitfunctions/pull_branch.py matteo_branch

# 📘 Git Workflow Guide for ML Projects

Welcome to your personalized Git cheat sheet! Since you are working on evaluating models and fine-tuning them (e.g., using LoRA methods from the `eomt` experiments), you will be writing lots of code, testing hypotheses, and generating model weights.

This guide will show you how to safely manage your work on your dedicated branch (`matteo_branch`).

---

## 1️⃣ Saving Your Work (Commit & Push)
Whenever you reach a good stopping point, finish writing a new evaluation script, or tweak a LoRA parameter, you should save your work to GitHub.

### Method A: The Easy Way (Using your helper script)
Run this command in a new code cell. It will add all your changes, label them with your message, and push them to `matteo_branch`.
```python
!python gitfunctions/update_branch.py "Added LoRA fine-tuning loop for model evaluation"
```

### Method B: The Raw Git Way (Without the script)
If you want to know what the script is doing behind the scenes, these are the standard Git commands:
```bash
%%bash
# 1. 'Stage' all modified and new files to be saved
git add .

# 2. 'Commit' (save) the files with a descriptive message
git commit -m "Added LoRA fine-tuning loop for model evaluation"

# 3. 'Push' (upload) the saved changes to your branch on GitHub
git push origin matteo_branch
```

## 2️⃣ Getting the Latest Updates (Pull)
If you edited files on a different computer, or if a colleague updated a shared file, you need to pull those changes into your current Colab environment.

### Method A: The Easy Way
```python
# Pull updates specifically for your branch
!python gitfunctions/pull_branch.py matteo_branch
```

### Method B: The Raw Git Way
```bash
%%bash
# 1. Fetch the latest information from GitHub
git fetch origin

# 2. Make sure you are on your branch
git checkout matteo_branch

# 3. Download and merge the changes
git pull origin matteo_branch
```

## 3️⃣ Branching for Experiments (e.g., LoRA Fine-Tuning)
In Machine Learning, you often want to try a crazy new idea (like a new LoRA config) without breaking your working code. You do this by creating a **new branch** branching off from `matteo_branch`.

### Creating a new experiment branch
Let's say you want to try an experiment with adapter weights.
```bash
%%bash
# Create and switch to a new branch called 'matteo_lora_exp'
git checkout -b matteo_lora_exp
```
Now you can change files, train your model, and push normally. If the experiment fails, you can easily switch back to your safe branch and delete the experiment:
```bash
%%bash
# Switch back to your safe branch
git checkout matteo_branch

# (Optional) Delete the failed experiment branch locally
git branch -d matteo_lora_exp
```

## 4️⃣ Recovering History & Undoing Mistakes
Everyone makes mistakes! Here is how to fix common Git problems.

### Scenario A: "I changed some files but I hate the changes. I want to go back to how they were at my last commit!"
```bash
%%bash
# This throws away ALL uncommitted changes! Be careful!
git restore .
```

### Scenario B: "I want to look at the history of my commits"
```bash
%%bash
# Shows a neat list of your past commits and their ID numbers (hashes)
git log --oneline -n 5
```

### Scenario C: "I committed something, but I want to undo that commit (without losing the file changes)"
```bash
%%bash
# This undoes the last commit, but keeps your files exactly as they are right now
git reset --soft HEAD~1
```

## 5️⃣ Handling Large Machine Learning Files ⚠️
Since you are evaluating models and using LoRA, you will be generating **Large Files**:
*   `.safetensors`, `.bin`, `.pt`, `.pth` (Model Weights)
*   Large `.csv` or `.json` (Datasets)

**CRITICAL RULE:** NEVER run `git add .` if you have large model files in your folder, unless you are sure they are ignored by your `.gitignore` file. GitHub has a strict 100MB file limit. If you push a large model weight, it will freeze your repository.

**How to handle large files:**
1. Check your `.gitignore` file and ensure `*.safetensors` and `*.pth` are listed in it.
2. Save your trained models directly to Google Drive (e.g., in a dedicated `Saved_Models/` folder outside of the Git repository) instead of pushing them to GitHub.

In [7]:
# Run this cell to easily commit and push your current changes to 'matteo_branch'
# Feel free to change the commit message in the quotes below.
!python gitfunctions/update_branch.py "Update folder disposition"

--- Starting Update Process ---
> git add .
> git commit -m "Update folder disposition"
[main 3b0497d] Update folder disposition
 50 files changed, 47369 insertions(+), 59 deletions(-)
 delete mode 100644 coco-classes-mapping-master/coco_mapping_80to91.json
 delete mode 100644 coco-classes-mapping-master/coco_mapping_91to80.json
 create mode 100644 eomt/.gitignore
 create mode 100644 eomt/LICENSE
 create mode 100644 eomt/README.md
 rename =4.27.7 => eomt/__init__.py (100%)
 rename {configs => eomt/configs}/dinov2/cityscapes/semantic/eomt_base_640.yaml (100%)
 rename {configs => eomt/configs}/dinov2/coco/panoptic/eomt_base_640_2x.yaml (100%)
 create mode 100644 eomt/docs/index.html
 create mode 100644 eomt/docs/static/css/bulma-carousel.min.css
 create mode 100644 eomt/docs/static/css/bulma-slider.min.css
 create mode 100644 eomt/docs/static/css/bulma.css.map.txt
 create mode 100644 eomt/docs/static/css/bulma.min.css
 create mode 100644 eomt/docs/static/css/fontawesome.all.min.css
 crea

In [3]:
import os
from google.colab import userdata

# Set your Git identity
!git config --global user.email "s360426@studenti.unipi.it"
!git config --global user.name "MatteoAldovardi92"

# Retrieve the GitHub token from Colab Secrets
github_token = userdata.get('GITHUB_TOKEN')

# Securely configure Git to use the token for all GitHub interactions
os.system(f'git config --global url."https://{github_token}@github.com/".insteadOf "https://github.com/"')

print("✅ Git identity and token configured successfully! You can now run the update cell.")

✅ Git identity and token configured successfully! You can now run the update cell.


In [5]:
%%writefile gitfunctions/update_branch.py
import os
import sys
import subprocess

def run_cmd(cmd):
    print(f"> {cmd}")
    os.system(cmd)

if __name__ == "__main__":
    commit_message = "Update from Colab"
    if len(sys.argv) > 1:
        commit_message = sys.argv[1]

    print("--- Starting Update Process ---")
    run_cmd("git add .")
    run_cmd(f'git commit -m "{commit_message}"')

    # Automatically get the current active branch
    current_branch = subprocess.getoutput("git rev-parse --abbrev-ref HEAD")

    print(f"Pushing to branch: {current_branch}")
    run_cmd(f"git push origin {current_branch}")
    print("--- Update Complete! ---")

Overwriting gitfunctions/update_branch.py


In [6]:
# 1. Unstage everything
!git reset

# 2. Append large file rules to .gitignore
with open('.gitignore', 'a') as f:
    f.write('\n# Ignore large ML files and datasets\n')
    f.write('*.safetensors\n*.bin\n*.pt\n*.pth\n*.h5\n*.csv\n*.json\nSaved_Models/\n')

# 3. Clear git cache so it respects the updated .gitignore for all files
!git rm -r --cached .

print('✅ Slate cleaned and .gitignore updated! Large files will now be ignored.\n➡️ You can now run the update cell above to push your changes.')

Unstaged changes after reset:
D	=4.27.7
D	configs/dinov2/cityscapes/semantic/eomt_base_640.yaml
D	configs/dinov2/coco/panoptic/eomt_base_640_2x.yaml
D	generate_mapping.py
M	gitfunctions/main.ipynb
M	gitfunctions/update_branch.py

It took 20.90 seconds to enumerate unstaged changes after reset.  You can
use '--quiet' to avoid this.  Set the config setting reset.quiet to true
to make this the default.
rm '.gitignore'
rm '=4.27.7'
rm 'LICENSE'
rm 'README.md'
rm 'Step4.ipynb'
rm 'Step4_Professional_Cleaned.ipynb'
rm 'coco-classes-mapping-master/README.md'
rm 'coco-classes-mapping-master/coco80.names'
rm 'coco-classes-mapping-master/coco91.names'
rm 'coco-classes-mapping-master/coco_mapping_80to91.json'
rm 'coco-classes-mapping-master/coco_mapping_91to80.json'
rm 'coco-classes-mapping-master/map_coco_classes.py'
rm 'configs/dinov2/cityscapes/semantic/eomt_base_640.yaml'
rm 'configs/dinov2/coco/panoptic/eomt_base_640_2x.yaml'
rm 'docs/index.html'
rm 'docs/static/css/bulma-carousel.min.css'
r

In [10]:
import os

# 1. Create the new subfolder and navigate into it
os.makedirs('eomt/data', exist_ok=True)
# Save current directory to return later
base_dir = os.getcwd()
os.chdir('eomt/data')

# 2. Set credentials as environment variables for safe string handling
os.environ['CS_USER'] = "s360426@studenti.polito.it"
os.environ['CS_PASS'] = "UserProdRDD1!s"

print("Logging into Cityscapes...")
# 3. Login and save cookies
!wget -q --show-progress --keep-session-cookies --save-cookies=cookies.txt --post-data "username=$CS_USER&password=$CS_PASS&submit=Login" https://www.cityscapes-dataset.com/login/

print("\nDownloading Package 1 (gtFine_trainvaltest.zip)...")
# 4. Download first dataset
!wget -q --show-progress --load-cookies cookies.txt --content-disposition https://www.cityscapes-dataset.com/file-handling/?packageID=1

print("\nDownloading Package 3 (leftImg8bit_trainvaltest.zip - ~11GB)...")
# 5. Download second dataset
!wget -q --show-progress --load-cookies cookies.txt --content-disposition https://www.cityscapes-dataset.com/file-handling/?packageID=3

# 6. Clean up the cookies file
if os.path.exists('cookies.txt'):
    os.remove('cookies.txt')

# Return to base directory
os.chdir(base_dir)
print("\n✅ Downloads completed in the eomt/data folder!")

Logging into Cityscapes...
index.html.2            [  <=>               ]  55.82K   260KB/s    in 0.2s    

gtFine_trainvaltest 100%[===================>] 240.87M  14.7MB/s    in 16s     

leftImg8bit_trainva 100%[===================>]  10.80G  22.1MB/s    in 9m 35s  

✅ Downloads completed in the eomt/data folder!


In [14]:
import os
import glob

data_dir = 'eomt/data'

print("🔍 Checking files and sizes in eomt/data:")
for file in sorted(os.listdir(data_dir)):
    filepath = os.path.join(data_dir, file)
    size_mb = os.path.getsize(filepath) / (1024 * 1024)
    print(f" - {file}: {size_mb:.2f} MB")

print("\n🧹 Running cleanup...")

# 1. Remove any file ending in .1 (or .html.2 etc)
# Using set() to avoid duplicates if a file matches both patterns
for f in set(glob.glob(f'{data_dir}/*.1') + glob.glob(f'{data_dir}/index.html*')):
    if os.path.exists(f):
        os.remove(f)
        print(f"Deleted duplicate/temp file: {f}")

# 2. Remove the original .zip files (the incomplete ones)
for f in glob.glob(f'{data_dir}/*.zip'):
    if os.path.exists(f):
        os.remove(f)
        print(f"Deleted incomplete zip: {f}")

# 3. Rename .zip.2 files to .zip
for f in glob.glob(f'{data_dir}/*.zip.2'):
    if os.path.exists(f):
        new_name = f.replace('.zip.2', '.zip')
        os.rename(f, new_name)
        print(f"Renamed: {f} -> {new_name}")

print("✅ Cleanup complete!")


🔍 Checking files and sizes in eomt/data:
 - gtFine_trainvaltest.zip: 240.87 MB
 - gtFine_trainvaltest.zip.2: 240.87 MB
 - index.html.2: 0.05 MB
 - leftImg8bit_trainvaltest.zip: 4676.86 MB
 - leftImg8bit_trainvaltest.zip.2: 11055.30 MB

🧹 Running cleanup...
Deleted duplicate/temp file: eomt/data/index.html.2
Deleted incomplete zip: eomt/data/gtFine_trainvaltest.zip
Deleted incomplete zip: eomt/data/leftImg8bit_trainvaltest.zip
Renamed: eomt/data/gtFine_trainvaltest.zip.2 -> eomt/data/gtFine_trainvaltest.zip
Renamed: eomt/data/leftImg8bit_trainvaltest.zip.2 -> eomt/data/leftImg8bit_trainvaltest.zip
✅ Cleanup complete!


In [25]:
import os
import json

file_path = 'Step4.ipynb'

if os.path.exists(file_path):
    print(f"🔍 File '{file_path}' exists. Size: {os.path.getsize(file_path)} bytes")
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            notebook = json.load(f)
        print("✅ The file is valid JSON. The corruption might be inside a specific cell's structure.")
    except json.JSONDecodeError as e:
        print(f"❌ JSONDecodeError: The file is NOT a valid notebook. Error: {e}")
        print("\n--- First 500 characters of the file ---")
        with open(file_path, 'r', encoding='utf-8', errors='replace') as f:
            print(f.read(500))
else:
    print(f"❌ File '{file_path}' not found in the current directory.")

🔍 File 'Step4.ipynb' exists. Size: 500153 bytes
✅ The file is valid JSON. The corruption might be inside a specific cell's structure.


In [26]:
import json
import os

file_path = 'Step4.ipynb'
fixed_path = 'Step4_fixed.ipynb'

with open(file_path, 'r', encoding='utf-8') as f:
    nb = json.load(f)

cleaned_cells = []
for i, cell in enumerate(nb.get('cells', [])):
    # Ensure minimum required fields for Jupyter notebooks
    if 'cell_type' not in cell:
        cell['cell_type'] = 'code'
    if 'source' not in cell:
        cell['source'] = []
    if 'metadata' not in cell:
        cell['metadata'] = {}

    # Strip outputs and execution counts to fix potential rendering crashes
    if cell['cell_type'] == 'code':
        cell['outputs'] = []
        cell['execution_count'] = None

    cleaned_cells.append(cell)

nb['cells'] = cleaned_cells

with open(fixed_path, 'w', encoding='utf-8') as f:
    json.dump(nb, f, indent=2)

print(f"✅ Notebook cleaned and saved as '{fixed_path}'.\n➡️ Please refresh the file browser on the left and try opening '{fixed_path}'!")

✅ Notebook cleaned and saved as 'Step4_fixed.ipynb'.
➡️ Please refresh the file browser on the left and try opening 'Step4_fixed.ipynb'!


In [15]:
import os

data_dir = 'eomt/data'

print("🔍 Final check of files and sizes in eomt/data:")
if os.path.exists(data_dir):
    for file in sorted(os.listdir(data_dir)):
        filepath = os.path.join(data_dir, file)
        size_mb = os.path.getsize(filepath) / (1024 * 1024)
        print(f" - {file}: {size_mb:.2f} MB")
else:
    print(f"Directory {data_dir} does not exist.")


🔍 Final check of files and sizes in eomt/data:
 - gtFine_trainvaltest.zip: 240.87 MB
 - leftImg8bit_trainvaltest.zip: 4676.86 MB


In [16]:
import os

data_dir = 'eomt/data'
bad_file = os.path.join(data_dir, 'leftImg8bit_trainvaltest.zip')

# 1. Delete the incomplete file
if os.path.exists(bad_file):
    os.remove(bad_file)
    print(f"🗑️ Deleted incomplete file: {bad_file}")

# Save current directory to return later
base_dir = os.getcwd()
os.chdir(data_dir)

# 2. Set credentials
os.environ['CS_USER'] = "s360426@studenti.polito.it"
os.environ['CS_PASS'] = "UserProdRDD1!s"

print("🔑 Logging into Cityscapes...")
# 3. Login and save cookies
!wget -q --show-progress --keep-session-cookies --save-cookies=cookies.txt --post-data "username=$CS_USER&password=$CS_PASS&submit=Login" https://www.cityscapes-dataset.com/login/

print("\n⬇️ Downloading Package 3 (leftImg8bit_trainvaltest.zip - ~11GB). This will take a while...")
# 4. Download Package 3
!wget -q --show-progress --load-cookies cookies.txt --content-disposition https://www.cityscapes-dataset.com/file-handling/?packageID=3

# 5. Clean up the cookies file
if os.path.exists('cookies.txt'):
    os.remove('cookies.txt')

# Return to base directory
os.chdir(base_dir)
print("\n✅ Download of Package 3 completed in the eomt/data folder!")

🗑️ Deleted incomplete file: eomt/data/leftImg8bit_trainvaltest.zip
🔑 Logging into Cityscapes...
index.html              [  <=>               ]  55.82K   261KB/s    in 0.2s    

⬇️ Downloading Package 3 (leftImg8bit_trainvaltest.zip - ~11GB). This will take a while...
leftImg8bit_trainva 100%[===================>]  10.80G  26.8MB/s    in 7m 32s  

✅ Download of Package 3 completed in the eomt/data folder!


In [17]:
import os
import shutil

# Define paths
project_dir = '/content/drive/MyDrive/FundGitHubProject'
drive_data_dir = os.path.join(project_dir, 'eomt/data')
local_temp_dir = '/content/temp_download'

# Ensure directories exist
os.makedirs(drive_data_dir, exist_ok=True)
os.makedirs(local_temp_dir, exist_ok=True)

# Switch to local storage for the heavy download
os.chdir(local_temp_dir)

# Credentials
os.environ['CS_USER'] = "s360426@studenti.polito.it"
os.environ['CS_PASS'] = "UserProdRDD1!s"

print("🔑 Logging into Cityscapes (Local Environment)...")
!wget -q --show-progress --keep-session-cookies --save-cookies=cookies.txt --post-data "username=$CS_USER&password=$CS_PASS&submit=Login" https://www.cityscapes-dataset.com/login/

print("\n⬇️ Downloading Package 3 to LOCAL VM storage (This prevents Drive truncation)...")
!wget -q --show-progress --load-cookies cookies.txt --content-disposition https://www.cityscapes-dataset.com/file-handling/?packageID=3

local_file = 'leftImg8bit_trainvaltest.zip'
if os.path.exists(local_file):
    local_size = os.path.getsize(local_file) / (1024**3)
    print(f"\n✅ Local download complete. Size: {local_size:.2f} GB")

    print("\n🚚 Copying the 11GB file to Google Drive... (This might take a few minutes)")
    drive_path = os.path.join(drive_data_dir, local_file)

    # Using cp is often more reliable for large files in Colab than python's shutil
    !cp {local_file} "{drive_path}"

    if os.path.exists(drive_path):
        drive_size = os.path.getsize(drive_path) / (1024**3)
        print(f"✅ Copy complete. Drive file size: {drive_size:.2f} GB")

        if abs(local_size - drive_size) < 0.1:
            print("🎉 Success! The full 11GB file is now safely in your Drive.")
        else:
            print("⚠️ Warning: Sizes don't match exactly. Check the Drive folder.")
    else:
        print("❌ Failed to copy to Drive.")
else:
    print("❌ Download to local storage failed.")

# Cleanup local files to free up disk space on the VM
print("🧹 Cleaning up local temporary files...")
if os.path.exists(local_file):
    os.remove(local_file)
if os.path.exists('cookies.txt'):
    os.remove('cookies.txt')

# Return safely to project directory
os.chdir(project_dir)
print("Done!")

🔑 Logging into Cityscapes (Local Environment)...
index.html              [  <=>               ]  55.82K   250KB/s    in 0.2s    

⬇️ Downloading Package 3 to LOCAL VM storage (This prevents Drive truncation)...
leftImg8bit_trainva 100%[===================>]  10.80G  30.5MB/s    in 6m 17s  

✅ Local download complete. Size: 10.80 GB

🚚 Copying the 11GB file to Google Drive... (This might take a few minutes)
✅ Copy complete. Drive file size: 10.80 GB
🎉 Success! The full 11GB file is now safely in your Drive.
🧹 Cleaning up local temporary files...
Done!


In [18]:
import os

# Ensure we are in the project root
project_dir = '/content/drive/MyDrive/FundGitHubProject'
os.chdir(project_dir)

# Add .zip and the data folder to .gitignore
with open('.gitignore', 'a') as f:
    f.write('\n# Ignore compressed datasets and data folders\n')
    f.write('*.zip\neomt/data/\n')

# Clear git cache so it respects the new rules
print("🧹 Clearing git cache to apply new rules...")
!git rm -r --cached . > /dev/null

print("\n🔍 Checking git status (ensure .zip files are NOT listed under 'Untracked files'):")
!git status

print("\n✅ Slate cleaned! The 11GB file is now officially ignored. You can now run your git push cell.")

🧹 Clearing git cache to apply new rules...

🔍 Checking git status (ensure .zip files are NOT listed under 'Untracked files'):
On branch main
Your branch is up to date with 'origin/main'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	deleted:    .gitignore
	deleted:    LICENSE
	deleted:    README.md
	deleted:    Step4.ipynb
	deleted:    Step4_Professional_Cleaned.ipynb
	deleted:    coco-classes-mapping-master/README.md
	deleted:    coco-classes-mapping-master/coco80.names
	deleted:    coco-classes-mapping-master/coco91.names
	deleted:    coco-classes-mapping-master/map_coco_classes.py
	deleted:    docs/index.html
	deleted:    docs/static/css/bulma-carousel.min.css
	deleted:    docs/static/css/bulma-slider.min.css
	deleted:    docs/static/css/bulma.css.map.txt
	deleted:    docs/static/css/bulma.min.css
	deleted:    docs/static/css/fontawesome.all.min.css
	deleted:    docs/static/css/index.css
	deleted:    docs/static/images/apple-touch-icon.png
	deleted: 

In [19]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	deleted:    .gitignore
	deleted:    LICENSE
	deleted:    README.md
	deleted:    Step4.ipynb
	deleted:    Step4_Professional_Cleaned.ipynb
	deleted:    coco-classes-mapping-master/README.md
	deleted:    coco-classes-mapping-master/coco80.names
	deleted:    coco-classes-mapping-master/coco91.names
	deleted:    coco-classes-mapping-master/map_coco_classes.py
	deleted:    docs/index.html
	deleted:    docs/static/css/bulma-carousel.min.css
	deleted:    docs/static/css/bulma-slider.min.css
	deleted:    docs/static/css/bulma.css.map.txt
	deleted:    docs/static/css/bulma.min.css
	deleted:    docs/static/css/fontawesome.all.min.css
	deleted:    docs/static/css/index.css
	deleted:    docs/static/images/apple-touch-icon.png
	deleted:    docs/static/images/arch.svg
	deleted:    docs/static/images/favicon-96x96.png
	deleted:    docs/static/images/favicon.ico
	

In [20]:
!python gitfunctions/update_branch.py "Update .gitignore to ignore large files and datasets"

--- Starting Update Process ---
> git add .
> git commit -m "Update .gitignore to ignore large files and datasets"
[main 3627df6] Update .gitignore to ignore large files and datasets
 2 files changed, 5 insertions(+), 1 deletion(-)
Pushing to branch: main
> git push origin main
Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 2 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 8.94 KiB | 832.00 KiB/s, done.
Total 5 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To https://github.com/MatteoAldovardi92/Fundamental_Project.git
   3b0497d..3627df6  main -> main
--- Update Complete! ---


In [27]:
import os

# Ensure we are in the project root
project_dir = '/content/drive/MyDrive/FundGitHubProject'
os.chdir(project_dir)

# Define the content for each README file
readmes = {
    "README_AI_Guide.md": """# 🚀 Fundamental ML Project - Beginner's Guide

*Disclaimer: This document is AI generated.*

Welcome to the Fundamental Project! This repository is focused on Machine Learning model evaluation and fine-tuning (e.g., semantic segmentation, LoRA experiments, and anomaly detection).

## ⚠️ Important Note for Novices
**This repository is NOT self-contained.** To keep the repository lightweight and comply with Git limits, **large files have been excluded**.
- **Missing Datasets:** Datasets (like the 11GB Cityscapes data) are ignored. You must retrieve them locally into the `eomt/data/` folder using the provided download notebook cells.
- **Missing Model Weights:** Pre-trained model weights (`.pth`, `.safetensors`, etc.) are also excluded. Ensure you place your weights in the appropriate local directories before running inference.

## 📂 Repository Structure
- **`eval/`**: Contains scripts to evaluate model performance (IoU metrics, anomaly detection).
- **`eomt/`**: Contains the core model architectures, configurations, and training pipelines.
- **`gitfunctions/`**: Helper scripts designed to make Git commands easier from Google Colab.
- **`coco-classes-mapping-master/`**: Utilities for handling COCO dataset class indices.
- **`trained_models/`**: Designated storage for model weights.

Dive into each subfolder to find its specific `README_AI_Guide.md` for more details on the individual scripts!
""",

    "eval/README_AI_Guide.md": """# 📊 Evaluation Module (`eval/`)

*Disclaimer: This document is AI generated.*

This module contains all the necessary scripts to assess the performance of the trained computer vision models.

## 📝 Key Scripts:
- **`dataset.py` & `transform.py`**: Handles loading and preprocessing/augmenting the evaluation datasets (like Cityscapes).
- **`erfnet.py` & `erfnet_nobn.py`**: Defines the ERFNet architecture (Efficient Residual Factorized ConvNet) used for real-time semantic segmentation during evaluation.
- **`evalAnomaly.py`**: Script dedicated to detecting out-of-distribution or anomalous objects in the scene.
- **`eval_iou.py` & `iouEval.py`**: Calculates the Intersection over Union (IoU) metric, which is the standard measure for semantic segmentation accuracy.
- **`eval_cityscapes_color.py` & `eval_cityscapes_server.py`**: Specific evaluation pipelines tailored to the Cityscapes dataset formats.
""",

    "eomt/README_AI_Guide.md": """# 🧠 End-of-Module Training (`eomt/`)

*Disclaimer: This document is AI generated.*

This folder holds the core training logic, model architectures, and experimental setups (such as LoRA fine-tuning workflows) for the project.

## 📝 Folder Breakdown:
- **`models/`**: Defines the neural network architecture (e.g., Vision Transformers). Check `eomt/models/README_AI_Guide.md` for details.
- **`training/`**: Contains the PyTorch Lightning trainer logic and loss functions. Check `eomt/training/README_AI_Guide.md` for details.
- **`configs/`**: Stores `.yaml` configuration files that govern hyperparameters. Check `eomt/configs/README_AI_Guide.md` for details.
- **`datasets/`**: Handles the data loading pipelines for PyTorch. Check `eomt/datasets/README_AI_Guide.md` for details.
- **`docs/`**: Web assets for project showcase. Check `eomt/docs/README_AI_Guide.md` for details.
- **`data/`**: The designated folder for heavy datasets (e.g., Cityscapes). *Note: Contents of this folder are ignored by git to save space.*
- **`main.py`**: The primary entry point for kicking off a training or fine-tuning run.
""",

    "eomt/models/README_AI_Guide.md": """# 🏗️ Model Architectures (`eomt/models/`)

*Disclaimer: This document is AI generated.*

This directory defines the actual structure of the neural networks you are training or fine-tuning.

## 📝 Key Scripts:
- **`vit.py`**: Implements the Vision Transformer (ViT) backbone. This is often the foundational model that extracts features from your images.
- **`scale_block.py`**: Defines scaling modules or intermediate blocks (potentially used for adapter/LoRA integration) that adjust the features passing through the network.
- **`eomt.py`**: The overarching wrapper script that ties the backbone (like ViT) together with the task-specific heads (like a segmentation head) to build the complete, end-to-end model.
""",

    "eomt/training/README_AI_Guide.md": """# 🏋️ Training Logic & Lightning Modules (`eomt/training/`)

*Disclaimer: This document is AI generated.*

This folder abstracts the complex PyTorch boilerplate loops (like moving tensors to GPUs, zeroing gradients, and step tracking) using PyTorch Lightning.

## 📝 Key Scripts:
- **`lightning_module.py`**: The core PyTorch Lightning module. It defines what happens during `training_step`, `validation_step`, and configures the optimizers and learning rate schedulers.
- **`mask_classification_semantic.py`**: Defines the specific logic, loss calculations, and metrics needed for Semantic Segmentation (classifying every pixel into categories like 'road', 'car', 'tree').
- **`mask_classification_panoptic.py`**: Handles Panoptic Segmentation logic, which is a step beyond semantic—it not only identifies pixels but separates distinct instances of objects (e.g., 'car 1' vs 'car 2').
- **`mask_classification_instance.py`**: Handles pure Instance Segmentation tasks.
- **`mask_classification_loss.py`**: Centralizes the mathematical loss functions used to train the segmentation models.
- **`two_stage_warmup_poly_schedule.py`**: A custom learning rate scheduler often used in computer vision to slowly warm up the learning rate before decaying it.
""",

    "eomt/configs/README_AI_Guide.md": """# ⚙️ Training Configurations (`eomt/configs/`)

*Disclaimer: This document is AI generated.*

Instead of hardcoding batch sizes, learning rates, and dataset paths directly into Python scripts, machine learning projects use configuration files. This makes running multiple different experiments much easier.

## 📝 How it works:
- You will typically find `.yaml` files in subdirectories here (e.g., `dinov2/cityscapes/semantic/eomt_base_640.yaml`).
- When you run a training script (like `main.py`), you will pass one of these config files as an argument.
- These configs define:
  - Which dataset to use (e.g., Cityscapes).
  - The image resolution (e.g., 640x640).
  - Training hyperparameters (epochs, learning rate, weight decay).
  - Model specific settings (ViT base vs ViT large).
""",

    "eomt/datasets/README_AI_Guide.md": """# 🗄️ Dataset Loaders & Processing (`eomt/datasets/`)

*Disclaimer: This document is AI generated.*

This directory is absolutely critical for feeding data into your model properly. It handles the loading, parsing, and preprocessing of your raw datasets (like Cityscapes or COCO) before the tensors reach the neural network.

## 📝 Key Responsibilities:
- **PyTorch Dataset Classes (`Dataset`):** Custom classes that instruct PyTorch exactly how to read images and their corresponding ground-truth mask/label files from the disk directory.
- **Data Augmentation & Transforms:** Implementation of spatial transformations (like random cropping, resizing, horizontal flipping) and color jittering. This forces the model to learn robust features rather than memorizing exact pixels.
- **Label Mapping:** Converting raw pixel IDs from datasets into 'train IDs' used by the loss functions. For example, Cityscapes has 35 raw classes, but models are typically trained on a condensed set of 19 classes. This folder handles that ID translation.
- **Collation:** Preparing and padding batches of tensors (images, targets, and metadata) so they can be seamlessly processed by PyTorch `DataLoader` workers during the training loop.
""",

    "eomt/docs/README_AI_Guide.md": """# 🌐 Web Showcase & Documentation (`eomt/docs/`)

*Disclaimer: This document is AI generated.*

This folder contains the static web assets (HTML, CSS, JavaScript, and images) used to build a presentation page or showcase for the project.

## 📝 Key Components:
- **`index.html`**: The main webpage structure.
- **`static/css/` & `static/js/`**: Contains Bulma framework files (a lightweight CSS framework) and custom scripts for image carousels and layout.
- **`static/images/`**: Diagrams, architecture plots (`arch.svg`), and icons used on the webpage.
- **Purpose**: Typically, a folder structured like this is served using **GitHub Pages** to provide a user-friendly overview of the model's architecture, results, and anomaly detection capabilities without requiring users to read the code.
""",

    "coco-classes-mapping-master/README_AI_Guide.md": """# 🗺️ COCO Classes Mapping (`coco-classes-mapping-master/`)

*Disclaimer: This document is AI generated.*

This folder contains specialized utilities for handling the often-confusing MS COCO dataset class indices, which is critical for evaluating object detection and instance/panoptic segmentation models.

## 📝 What this does & Why it matters:
The original MS COCO dataset annotation file contains 91 categories. However, most modern models are trained on a condensed subset of 80 categories (excluding classes that have very few annotations, like 'hat', 'shoe', or 'window').
- **Mapping Scripts:** Scripts like `map_coco_classes.py` translate model predictions back and forth between the 91-class format (needed for official COCO evaluation servers) and the 80-class format (used during actual PyTorch training).
- **JSON & Name Maps:** Contains `.json` mapping dictionaries (`coco_mapping_80to91.json`) and `.names` text files for easy class ID lookups during debugging, evaluation, or visualization.
- **Preventing Errors:** Without these files, a model predicting class ID '10' in the 80-class system might be incorrectly evaluated as a completely different object in the 91-class official ground truth, ruining your mAP (mean Average Precision) scores.
""",

    "trained_models/README_AI_Guide.md": """# 💾 Pre-trained Models (`trained_models/`)

*Disclaimer: This document is AI generated.*

This directory is designated for storing serialized neural network weights (the actual "learned" knowledge of the models).

## 📝 Important Details:
- **Large Files Ignored:** To prevent Git from crashing, large model files (like `.pth`, `.safetensors`, `.bin`) are generally ignored by `.gitignore`.
- **Existing Files:** You might see smaller or essential base files here, such as `erfnet_encoder_pretrained.pth.tar`, which acts as a starting point (encoder backbone) before fine-tuning.
- **Workflow:** When you train a model or apply LoRA in this Colab environment, save your final `.pth` files here. However, remember to back them up to your personal Google Drive, as pushing massive files to GitHub will fail.
""",

    "gitfunctions/README_AI_Guide.md": """# 🛠️ Git Helper Functions (`gitfunctions/`)

*Disclaimer: This document is AI generated.*

Since this project is primarily developed inside Google Colab, standard terminal Git workflows can be cumbersome. This folder provides Python scripts to simplify interacting with GitHub.

## 📝 Key Scripts:
- **`update_branch.py`**: A one-click script to stage all changes, commit them with a message, and push them to the active branch. It automatically detects your current branch.
- **`pull_branch.py`**: Quickly pulls the latest changes from the remote repository to ensure your Colab environment is up to date.
"""
}

# Create the files
print("📝 Generating detailed README files (including docs & trained_models)...")
for filepath, content in readmes.items():
    # Ensure the directory exists (for subfolders)
    os.makedirs(os.path.dirname(filepath) if os.path.dirname(filepath) else '.', exist_ok=True)

    # Write the content to the file
    with open(filepath, 'w') as f:
        f.write(content)
    print(f" ✅ Created: {filepath}")

print("\n🎉 All detailed guide files have been successfully generated!")

📝 Generating detailed README files (including docs & trained_models)...
 ✅ Created: README_AI_Guide.md
 ✅ Created: eval/README_AI_Guide.md
 ✅ Created: eomt/README_AI_Guide.md
 ✅ Created: eomt/models/README_AI_Guide.md
 ✅ Created: eomt/training/README_AI_Guide.md
 ✅ Created: eomt/configs/README_AI_Guide.md
 ✅ Created: eomt/datasets/README_AI_Guide.md
 ✅ Created: eomt/docs/README_AI_Guide.md
 ✅ Created: coco-classes-mapping-master/README_AI_Guide.md
 ✅ Created: trained_models/README_AI_Guide.md
 ✅ Created: gitfunctions/README_AI_Guide.md

🎉 All detailed guide files have been successfully generated!


In [ ]:
%%writefile gitfunctions/pull_branch.py
import os
import argparse

def run_cmd(cmd):
    print(f"> {cmd}")
    os.system(cmd)

if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Pull latest changes from GitHub.")
    # Make branch an optional positional argument defaulting to 'main'
    parser.add_argument("branch", nargs="?", default="main", help="Target branch to pull (default: main)")
    args = parser.parse_args()

    print(f"--- Pulling latest changes for branch: {args.branch} ---")
    run_cmd("git fetch origin")
    run_cmd(f"git checkout {args.branch}")
    run_cmd(f"git pull origin {args.branch}")
    print("--- Pull Complete! ---")


In [ ]:
%%writefile gitfunctions/update_branch.py
import os
import argparse

def run_cmd(cmd):
    print(f"> {cmd}")
    os.system(cmd)

if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Commit and push changes to GitHub.")
    # Message is an optional positional argument
    parser.add_argument("message", nargs="?", default="Update from Colab", help="Commit message")
    # Branch is an optional named argument defaulting to 'main'
    parser.add_argument("--branch", "-b", default="main", help="Target branch to push to (default: main)")
    args = parser.parse_args()

    print("--- Starting Update Process ---")
    run_cmd("git add .")
    run_cmd(f'git commit -m "{args.message}"')

    print(f"Pushing to branch: {args.branch}")
    run_cmd(f"git push origin {args.branch}")
    print("--- Update Complete! ---")


### 🚀 Final Push
Run the cell below to commit all your new `README_AI_Guide.md` files and the updated Git scripts to your `main` branch!

In [ ]:
# Execute the final push to save all documentation and script updates
!python gitfunctions/update_branch.py "Add detailed AI guides and improve git helper scripts" --branch main